<a href="https://colab.research.google.com/github/fantaxiah/vesicles-yolo/blob/main/JUNG_Copy_%EF%BC%92_of_vesicles_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Requirements

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.7 MB/s eta 0:00:00
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conf

In [ ]:
!pip install -q git+https://github.com/sunsmarterjie/yolov12.git roboflow supervision

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 126.2 MB/s eta 0:00:00


In [ ]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 32.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44286 sha256=376801a2b1a53493223e6aec804fb9205d875f8fee128b2576b42a30ac6484e5
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [ ]:
!pip install -q albumentations==1.4.7

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 5.8 MB/s eta 0:00:00


In [ ]:
#!pip uninstall -y torch torchvision torchaudio
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
#!pip uninstall -y torch torchvision torchaudio
#!pip cache purge

In [ ]:
#!pip install torch==2.10.0 torchvision==0.17.2 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import os
from google.colab import userdata
from google.colab import drive
import numpy as np
import torch
import torch.nn as nn
from torchvision.transforms import ToTensor
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import FasterRCNNBoxScoreTarget
import cv2
import os
import yaml
import random
import shutil
from ultralytics import YOLO
import supervision as sv
from roboflow import Roboflow
import matplotlib.pyplot as plt
from pathlib import Path
import albumentations as A


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.


# Dataset

In [ ]:
# Load the API key from Colab userdata and set it as an environment variable
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

# Verify the API key is set
if "ROBOFLOW_API_KEY" not in os.environ or os.environ["ROBOFLOW_API_KEY"] is None:
    raise ValueError("ROBOFLOW_API_KEY environment variable is still not set after attempting to load from userdata.")
else:
  print("Roboflow API key loaded and environment variable set.")

NameError: name 'userdata' is not defined

In [ ]:
rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))
project = rf.workspace("myrines-workspace").project("vesicles-ipt-cy5-2")
dataset = project.version(1).download("yolov8")  # or yolov5, yolov7, etc.

In [ ]:
!ls {dataset.location}

In [ ]:
!cat {dataset.location}/data.yaml

In [ ]:
# # 'cmd' + '/' to comment and uncomment blocks of code
# base = "/content/vesicles-ipt-cy5-2-1"
# all_img_dir = os.path.join(base, "all_data", "images")
# all_lbl_dir = os.path.join(base, "all_data", "labels")

# os.makedirs(all_img_dir, exist_ok=True)
# os.makedirs(all_lbl_dir, exist_ok=True)

# for split in ["train_split", "val_split"]:
#     img_src = os.path.join(base, split, "images")
#     lbl_src = os.path.join(base, split, "labels")

#     for f in os.listdir(img_src):
#         shutil.copy2(os.path.join(img_src, f), os.path.join(all_img_dir, f))

#     for f in os.listdir(lbl_src):
#         shutil.copy2(os.path.join(lbl_src, f), os.path.join(all_lbl_dir, f))

# print("Merged images:", len(os.listdir(all_img_dir)))
# print("Merged labels:", len(os.listdir(all_lbl_dir)))

In [ ]:
version_name = "vesicles-ipt-cy5-2-1"
data_yaml_path = "/content/{}/data.yaml".format(version_name)

with open(data_yaml_path, "r") as f:
    data_yaml_content = yaml.safe_load(f)

# SETUP the yaml file to the correct file locations
data_yaml_content["train"] = "./train/images"
data_yaml_content["val"] = "./train/images"
if "test" in data_yaml_content:
    del data_yaml_content["test"]

with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)

!cat {data_yaml_path}

# Image Augmentation

In [ ]:
# Root of the dataset (from Roboflow download)
root = Path(dataset.location)
print(root)
# We'll draw examples from train/images
train_img_dir = root / "train" / "images"
all_train_imgs = sorted(list(train_img_dir.glob("*.jpg")) + list(train_img_dir.glob("*.png")))

# Define an augmentation pipeline
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
  ])

def show_augmented_examples(n_examples=3):
    """Show original vs augmented vesicle images."""
    if len(all_train_imgs) == 0:
        print("No images found in train/images.")
        return

    else:
      print(f"Found {len(all_train_imgs)} training images")

    sample_paths = random.sample(all_train_imgs, min(n_examples, len(all_train_imgs)))

    plt.figure(figsize=(8, 4 * len(sample_paths)))

    for i, img_path in enumerate(sample_paths):
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        aug = augment(image=img_rgb)["image"]

        # original
        plt.subplot(len(sample_paths), 2, 2*i + 1)
        plt.imshow(img_rgb)
        plt.title(f"Original\n{img_path.name}")
        plt.axis("off")

        # augmented
        plt.subplot(len(sample_paths), 2, 2*i + 2)
        plt.imshow(aug)
        plt.title("Augmented")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

# Run this to see a few examples:
show_augmented_examples(4)


# YOLO Architecture

In [ ]:
model = YOLO("yolov8n.pt")
# raw_model = model.model

In [ ]:
results = model.val(
  data=data_yaml_path,
  epochs=300,
  # batch=8,
  # patience=20,
  # hsv_h=0.015,
  # hsv_s=0.7,
  # hsv_v=0.4,
  # translate=0.1,
  # scale=0.5,
  # fliplr=0.5,
  # mosaic=1.0,
  # mixup=0.5,
  # degrees=90,
  # shear=0.0,
  # perspective=0.0,
  # copy_paste=0.0
)

In [ ]:
results = model.train(
    data=data_yaml_path,
    epochs=300,
    imgsz=1000
)

In [ ]:
!ls runs/detect/train/weights
print(results.save_dir)
print(dataset.location)

results_file_path = results.save_dir

## Image Predictions

In [ ]:
#class ImageScore(nn.Module):
#  def __init__(self, yolo_model):
#    super().__init__()
#    self.model = yolo_model

#  def forward(self, x):
#    preds = self.model(x)[0]
#    class_scores = preds[..., 4:]      #* preds[..., 5:].sigmoid()
#    max_conf = class_scores.max(dim=-1).values
#    image_score = max_conf.sum(dim=1)
#    return image_score.unsqueeze(-1)

In [ ]:
ds = sv.DetectionDataset.from_yolo(
    images_directory_path=f"{dataset.location}/train/images",
    annotations_directory_path=f"{dataset.location}/train/labels",
    data_yaml_path=f"{dataset.location}/data.yaml"
)

In [ ]:
best_file_path = f"{results_file_path}/weights/best.pt"
model = YOLO(best_file_path)

In [ ]:
image_path, image, target = ds[0]

results = model(image, conf=0.4, verbose=False)[0]
detections = sv.Detections.from_ultralytics(results)

print("Number of detections:", len(detections))
print("Confidences:", detections.confidence)

In [ ]:
from glob import glob


for i in range(len(ds)):
  image_path, image, target = ds[i]

  # preds
  results = model(image, conf=0.4, verbose=False)[0]  # LOWER threshold
  detections = sv.Detections.from_ultralytics(results).with_nms()

  print(f"Image {i} → Detections:", len(detections))  # DEBUG

  box_annotator = sv.BoxAnnotator(thickness=2)
  #label_annotator = sv.LabelAnnotator(text_scale=0.5)

  # annotate pred
  annotated_pred = image.copy()
  annotated_pred = box_annotator.annotate(scene=annotated_pred, detections=detections)
  #annotated_pred = label_annotator.annotate(scene=annotated_pred, detections=detections)

  # annotate ground truth
  annotated_gt = image.copy()
  annotated_gt = box_annotator.annotate(scene=annotated_gt, detections=target)
  #annotated_gt = label_annotator.annotate(scene=annotated_gt, detections=target)

  # plots
  plt.figure(figsize=(12, 6))

  plt.subplot(1, 2, 1)
  plt.imshow(cv2.cvtColor(annotated_gt, cv2.COLOR_BGR2RGB))
  plt.title("Ground Truth")
  plt.axis("off")

  plt.subplot(1, 2, 2)
  plt.imshow(cv2.cvtColor(annotated_pred, cv2.COLOR_BGR2RGB))
  plt.title("Predicted")
  plt.axis("off")

  plt.tight_layout()
  plt.show()

In [ ]:
image_path, image, target = ds[0]

results = model(image, conf=0.05)[0]
detections = sv.Detections.from_ultralytics(results)

print("Detections:", len(detections))

In [ ]:
false_positive_dir = "false_positives"
os.makedirs(false_positive_dir, exist_ok=True)

for i in range(len(ds)):
  image_path, image, target = ds[i]

  if target is not None and "boxes" in target and len(target["boxes"]) > 0:
    continue

  # run inference
  #results = model(image, verbose=False)[0]
  results = model(image, conf=0.1, verbose=False)[0]
  detections = sv.Detections.from_ultralytics(results).with_nms()

  print("Detections:", len(detections))

  # ff model pred
  if len(detections.xyxy) > 0:
    annotated_image = image.copy()
    annotated_image = sv.BoxAnnotator().annotate(scene=annotated_image, detections=detections)
    #annotated_image = sv.LabelAnnotator().annotate(scene=annotated_image, detections=detections)

    save_path = os.path.join(false_positive_dir, f"fp_{i}.jpg")
    cv2.imwrite(save_path, annotated_image[..., ::-1])


In [ ]:
folder = "false_positives"
image_files = sorted(os.listdir(folder))

for filename in image_files:
    img_path = os.path.join(folder, filename)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.imshow(img)
    plt.title(filename)
    plt.axis('off')
    plt.show()

In [ ]:
!cat /content/vesicleimages-green-1/data.yaml


# Genetic Algorithm

In [ ]:
def fitness_function(hyperparameters):
  """
  Evaluates the fitness of a set of hyperparameters by training a YOLO model.

  Args:
    hyperparameters: A dictionary of hyperparameters for YOLO training.

  Returns:
    The mAP@0.5:0.95 score from the validation results.
  """
  # Load the YOLO model
  model = YOLO("yolov5s.pt")

  # Train the model with the given hyperparameters
  results = model.train(
      data=data_yaml_path,
      epochs=hyperparameters['epochs'],
      batch=hyperparameters['batch'],
      lr0=hyperparameters['lr0'],
      lrf=hyperparameters['lrf'],
      momentum=hyperparameters['momentum'],
      weight_decay=hyperparameters['weight_decay'],
      hsv_h=hyperparameters['hsv_h'],
      hsv_s=hyperparameters['hsv_s'],
      hsv_v=hyperparameters['hsv_v'],
      translate=hyperparameters['translate'],
      scale=hyperparameters['scale'],
      fliplr=hyperparameters['fliplr'],
      mosaic=hyperparameters['mosaic'],
      mixup=hyperparameters['mixup'],
      degrees=hyperparameters['degrees'],
      shear=hyperparameters['shear'],
      perspective=hyperparameters['perspective'],
      copy_paste=hyperparameters['copy_paste'],
      patience=hyperparameters['patience'],
      verbose=False # Set verbose to False to reduce output during training
  )

  # Extract mAP@0.5:0.95 from the results
  # The results object contains performance metrics. We need to access the mAP@0.5:0.95 metric.
  # Based on the ultralytics documentation and typical training output, this is often
  # available in results.results_dict['metrics/mAP_0.5:0.95(B)'].
  # However, the structure might vary slightly based on the ultralytics version.
  # Let's try to access it directly or look for a similar key.
  # A safer approach is to access the metrics property of the results object.
  # The metrics are usually stored in the `results.metrics` attribute after training.
  # According to the ultralytics documentation, mAP is accessed via `results.metrics.box.map`.

  try:
      # Access mAP@0.5:0.95 directly from the metrics
      map_0_5_95 = results.metrics.box.map
  except AttributeError:
      # If direct access fails, try to find it in the results dictionary
      try:
          map_0_5_95 = results.results_dict['metrics/mAP_0.5:0.95(B)']
      except KeyError:
          print("Could not find mAP@0.5:0.95 in training results.")
          map_0_5_95 = 0.0 # Return 0 if metric is not found

  return map_0_5_95

# Example usage of the fitness function (this part is for testing and not part of the final GA)
# Define a sample set of hyperparameters
# sample_hyperparameters = {
#     'epochs': 1,
#     'batch': 8,
#     'lr0': 0.01,
#     'lrf': 0.01,
#     'momentum': 0.937,
#     'weight_decay': 0.0005,
#     'hsv_h': 0.015,
#     'hsv_s': 0.7,
#     'hsv_v': 0.4,
#     'translate': 0.1,
#     'scale': 0.5,
#     'fliplr': 0.5,
#     'mosaic': 1.0,
#     'mixup': 0.0,
#     'degrees': 0.0,
#     'shear': 0.0,
#     'perspective': 0.0,
#     'copy_paste': 0.0,
#     'patience': 5
# }

# Call the fitness function with sample hyperparameters
# fitness_score = fitness_function(sample_hyperparameters)
# print(f"Fitness score for sample hyperparameters: {fitness_score}")

In [ ]:
def initialize_population(pop_size, ranges):
  """
  Initializes a population of hyperparameters.
  """
  population = []
  for _ in range(pop_size):
    hyperparameters = {}
    for key, value_range in ranges.items():
      if isinstance(value_range, tuple):
        if isinstance(value_range[0], int):
          hyperparameters[key] = random.randint(value_range[0], value_range[1])
        else:
          hyperparameters[key] = random.uniform(value_range[0], value_range[1])
      elif isinstance(value_range, list):
        hyperparameters[key] = random.choice(value_range)
    population.append(hyperparameters)
  return population

def fitness_function(hyperparameters, data_yaml_path):
  """
  Evaluates the fitness of a set of hyperparameters by training a YOLO model.
  """
  # Load the YOLO model
  model = YOLO("yolov5s.pt")

  # Train the model with the given hyperparameters
  results = model.train(
      data=data_yaml_path,
      epochs=hyperparameters['epochs'],
      batch=hyperparameters['batch'],
      lr0=hyperparameters['lr0'],
      lrf=hyperparameters['lrf'],
      momentum=hyperparameters['momentum'],
      weight_decay=hyperparameters['weight_decay'],
      hsv_h=hyperparameters['hsv_h'],
      hsv_s=hyperparameters['hsv_s'],
      hsv_v=hyperparameters['hsv_v'],
      translate=hyperparameters['translate'],
      scale=hyperparameters['scale'],
      fliplr=hyperparameters['fliplr'],
      mosaic=hyperparameters['mosaic'],
      mixup=hyperparameters['mixup'],
      degrees=hyperparameters['degrees'],
      shear=hyperparameters['shear'],
      perspective=hyperparameters['perspective'],
      copy_paste=hyperparameters['copy_paste'],
      patience=hyperparameters['patience'],
      verbose=False # Set verbose to False to reduce output during training
  )

    # Extract mAP@0.5:0.95 from the results
  try:
      map_0_5_95 = results.metrics.box.map
  except AttributeError:
      try:
          map_0_5_95 = results.results_dict['metrics/mAP_0.5:0.95(B)']
      except KeyError:
          print("Could not find mAP@0.5:0.95 in training results.")
          map_0_5_95 = 0.0

  return map_0_5_95

In [ ]:
def select_parents(population, num_parents):
  """
  Selects parent individuals from the population using tournament selection.
  """
  parents = []
  population_size = len(population)
  tournament_size = 3
  for _ in range(num_parents):
    tournament_indices = random.sample(range(population_size), tournament_size)
    tournament_candidates = [population[i] for i in tournament_indices]
    best_candidate = max(tournament_candidates, key=lambda item: item[1])
    parents.append(best_candidate[0])
  return parents

In [ ]:
def crossover(parent1_hyperparameters, parent2_hyperparameters):
  """
  Performs crossover between two parent hyperparameters to create offspring.
  """
  offspring_hyperparameters = {}
  for key in parent1_hyperparameters.keys():
    if random.random() < 0.5:
      offspring_hyperparameters[key] = parent1_hyperparameters[key]
    else:
      offspring_hyperparameters[key] = parent2_hyperparameters[key]
    if isinstance(parent1_hyperparameters[key], int):
        offspring_hyperparameters[key] = int(offspring_hyperparameters[key])
  return offspring_hyperparameters

In [ ]:
def mutate(hyperparameters, mutation_rate, mutation_strength):
  """
  Mutates the hyperparameters of an individual.
  """
  mutated_hyperparameters = hyperparameters.copy()
  for key in mutated_hyperparameters.keys():
    if random.random() < mutation_rate:
      if isinstance(mutated_hyperparameters[key], (int, float)):
        noise = random.gauss(0, mutation_strength)
        mutated_hyperparameters[key] += noise
        if key in ['lr0', 'lrf']:
            mutated_hyperparameters[key] = max(0.0001, min(0.1, mutated_hyperparameters[key]))
        elif key == 'momentum':
            mutated_hyperparameters[key] = max(0.8, min(0.99, mutated_hyperparameters[key]))
        elif key == 'weight_decay':
            mutated_hyperparameters[key] = max(0.0, min(0.001, mutated_hyperparameters[key]))
        elif key == 'epochs':
            mutated_hyperparameters[key] = int(max(10, min(500, mutated_hyperparameters[key])))
        elif key == 'batch':
             valid_batch_sizes = [4, 8, 16, 32, 64]
             current_batch = mutated_hyperparameters[key]
             try:
                 current_index = valid_batch_sizes.index(current_batch)
                 if random.random() < 0.5 and current_index > 0:
                     mutated_hyperparameters[key] = valid_batch_sizes[current_index - 1]
                 elif current_index < len(valid_batch_sizes) - 1:
                     mutated_hyperparameters[key] = valid_batch_sizes[current_index + 1]
             except ValueError:
                  mutated_hyperparameters[key] = random.choice(valid_batch_sizes)

        elif key in ['hsv_h', 'hsv_s', 'hsv_v', 'translate', 'scale', 'fliplr', 'mosaic', 'mixup', 'degrees', 'shear', 'perspective', 'copy_paste']:
             mutated_hyperparameters[key] = max(0.0, min(1.0, mutated_hyperparameters[key]))

  return mutated_hyperparameters

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
# Define parameters for the genetic algorithm
population_size = 10
num_generations = 50
mutation_rate = 0.1
mutation_strength = 0.05

# Define the range for hyperparameters
hyperparameter_ranges = {
    'epochs': (50, 300),
    'batch': [4, 8, 16, 32],
    'lr0': (0.001, 0.1),
    'lrf': (0.001, 0.1),
    'momentum': (0.8, 0.99),
    'weight_decay': (0.0, 0.001),
    'hsv_h': (0.0, 0.1),
    'hsv_s': (0.0, 1.0),
    'hsv_v': (0.0, 1.0),
    'translate': (0.0, 1.0),
    'scale': (0.0, 1.0),
    'fliplr': (0.0, 1.0),
    'mosaic': (0.0, 1.0),
    'mixup': (0.0, 1.0),
    'degrees': (0.0, 90.0),
    'shear': (0.0, 1.0),
    'perspective': (0.0, 0.001),
    'copy_paste': (0.0, 1.0),
    'patience': (10, 50)
}

# Initialize tracking variables
best_overall_fitness = -1.0
best_overall_hyperparameters = None
history = []

# Initialize the first population
population = initialize_population(population_size, hyperparameter_ranges)

# Start the main loop
for generation in range(num_generations):
  print(f"Generation {generation + 1}/{num_generations}")

  # Evaluate the fitness of each individual
  evaluated_population = []
  for hyperparameters in population:
    # Pass data_yaml_path to the fitness function
    fitness = fitness_function(hyperparameters, data_yaml_path)
    evaluated_population.append((hyperparameters, fitness))
    print(f"  Individual fitness: {fitness}")

  # Find the best-performing individual in this generation
  best_generation_individual = max(evaluated_population, key=lambda item: item[1])
  best_generation_hyperparameters, best_generation_fitness = best_generation_individual

  # Update the overall best
  if best_generation_fitness > best_overall_fitness:
      best_overall_fitness = best_generation_fitness
      best_overall_hyperparameters = best_generation_hyperparameters
      print(f"  New best overall fitness: {best_overall_fitness} with hyperparameters: {best_overall_hyperparameters}")
  else:
      print(f"  Best fitness in this generation: {best_generation_fitness}")

  # Append the generation's results to the history
  history.append({
      'generation': generation,
      'best_fitness': best_generation_fitness,
      'best_hyperparameters': best_generation_hyperparameters,
      'overall_best_fitness': best_overall_fitness,
      'overall_best_hyperparameters': best_overall_hyperparameters
  })

  # Select parents for the next generation
  num_parents = population_size // 2
  parents = select_parents(evaluated_population, num_parents)

  # Create offspring using crossover
  offspring = []
  num_offspring = population_size - 1
  for _ in range(num_offspring):
    parent1 = random.choice(parents)
    parent2 = random.choice(parents)
    child = crossover(parent1, parent2)
    offspring.append(child)

  # Mutate the offspring
  mutated_offspring = []
  for child in offspring:
      mutated_child = mutate(child, mutation_rate, mutation_strength)
      mutated_offspring.append(mutated_child)

  # Create the new generation (with elitism)
  new_population = [best_generation_hyperparameters]
  new_population.extend(mutated_offspring)

  while len(new_population) < population_size:
      new_population.append(random.choice(mutated_offspring))

  population = new_population

# After the loop finishes, print the best-performing hyperparameters
print("\nGenetic Algorithm Finished.")
print(f"Best overall fitness: {best_overall_fitness}")
print(f"Best overall hyperparameters: {best_overall_hyperparameters}")

# Pixels to um

# Size Distribution

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# microscope calibration – set correctly when you know it
PIXEL_SIZE_UM = 1.0  # µm per pixel

root = Path(dataset.location)
print("Dataset root:", root)


In [ ]:
# --- Cell A: size distribution from GT labels (train_split) ---

img_dir = root / "train_split" / "images"
lbl_dir = root / "train_split" / "labels"

img_files = sorted(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
print(f"Found {len(img_files)} images in train_split/images")

records = []

for img_path in img_files:
    # find matching label file
    stem = img_path.stem
    lbl_path = lbl_dir / f"{stem}.txt"
    if not lbl_path.exists():
        continue

    img = cv2.imread(str(img_path))
    if img is None:
        continue

    h, w = img.shape[:2]

    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cls_id = int(parts[0])
            x_c_n, y_c_n, w_n, h_n = map(float, parts[1:5])  # normalized

            # convert to pixels
            w_px = w_n * w
            h_px = h_n * h
            diameter_px = (w_px + h_px) / 2.0
            radius_px = diameter_px / 2.0
            area_px2 = np.pi * (radius_px ** 2)

            # convert to microns
            diameter_um = diameter_px * PIXEL_SIZE_UM
            radius_um = radius_px * PIXEL_SIZE_UM
            area_um2 = area_px2 * (PIXEL_SIZE_UM ** 2)

            records.append({
                "source": "train_split_GT",
                "image": img_path.name,
                "class_id": cls_id,
                "width_px": w_px,
                "height_px": h_px,
                "diameter_px": diameter_px,
                "radius_px": radius_px,
                "area_px2": area_px2,
                "diameter_um": diameter_um,
                "radius_um": radius_um,
                "area_um2": area_um2,
            })

if not records:
    print("No GT boxes found.")
else:
    df_train_gt = pd.DataFrame(records)
    print(f"GT detections (train_split): {len(df_train_gt)}")
    display(df_train_gt.head())

    # histogram
    plt.figure(figsize=(6,4))
    plt.hist(df_train_gt["diameter_px"], bins=30)
    plt.xlabel("Vesicle diameter (pixels)")
    plt.ylabel("Count")
    plt.title("Size distribution – GT (train_split)")
    plt.tight_layout()
    plt.show()


In [ ]:
# --- Cell B: size distribution from YOLO predictions on augmented images ---

aug_dir = root / "train_aug_images"

if not aug_dir.exists():
    print("No augmented images folder found:", aug_dir)
else:
    aug_imgs = sorted(list(aug_dir.glob("*.jpg")) + list(aug_dir.glob("*.png")))
    print(f"Found {len(aug_imgs)} augmented images.")

    records_aug = []

    for img_path in aug_imgs:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        results = model(img, verbose=False)[0]
        if results.boxes is None or len(results.boxes) == 0:
            continue

        boxes_xyxy = results.boxes.xyxy.cpu().numpy()
        confidences = results.boxes.conf.cpu().numpy()
        class_ids = results.boxes.cls.cpu().numpy().astype(int)

        for box, conf, cls_id in zip(boxes_xyxy, confidences, class_ids):
            x1, y1, x2, y2 = box
            w_px = x2 - x1
            h_px = y2 - y1
            diameter_px = (w_px + h_px) / 2.0
            radius_px = diameter_px / 2.0
            area_px2 = np.pi * (radius_px ** 2)

            diameter_um = diameter_px * PIXEL_SIZE_UM
            radius_um = radius_px * PIXEL_SIZE_UM
            area_um2 = area_px2 * (PIXEL_SIZE_UM ** 2)

            records_aug.append({
                "source": "augmented_pred",
                "image": img_path.name,
                "class_id": cls_id,
                "confidence": conf,
                "width_px": w_px,
                "height_px": h_px,
                "diameter_px": diameter_px,
                "radius_px": radius_px,
                "area_px2": area_px2,
                "diameter_um": diameter_um,
                "radius_um": radius_um,
                "area_um2": area_um2,
            })

    if not records_aug:
        print("No detections on augmented images.")
    else:
        df_aug_pred = pd.DataFrame(records_aug)
        print(f"Detections (augmented images): {len(df_aug_pred)}")
        display(df_aug_pred.head())

        plt.figure(figsize=(6,4))
        plt.hist(df_aug_pred["diameter_px"], bins=30)
        plt.xlabel("Vesicle diameter (pixels)")
        plt.ylabel("Count")
        plt.title("Size distribution – YOLO predictions (augmented images)")
        plt.tight_layout()
        plt.show()


In [ ]:
# --- Cell C: size distribution from YOLO predictions on full dataset ---

image_dirs = [
    root / "train_split" / "images",
    root / "val_split" / "images",
    root / "test" / "images",
]

# include augmented images if exist
if (root / "train_aug_images").exists():
    image_dirs.append(root / "train_aug_images")

all_imgs = []
for d in image_dirs:
    if d.exists():
        imgs = sorted(list(d.glob("*.jpg")) + list(d.glob("*.png")))
        all_imgs.extend(imgs)
        print(f"{d}: {len(imgs)} images")

print(f"\nTotal images for full-dataset analysis: {len(all_imgs)}")

records_all = []

for img_path in all_imgs:
    img = cv2.imread(str(img_path))
    if img is None:
        continue

    results = model(img, verbose=False)[0]
    if results.boxes is None or len(results.boxes) == 0:
        continue

    boxes_xyxy = results.boxes.xyxy.cpu().numpy()
    confidences = results.boxes.conf.cpu().numpy()
    class_ids = results.boxes.cls.cpu().numpy().astype(int)

    for box, conf, cls_id in zip(boxes_xyxy, confidences, class_ids):
        x1, y1, x2, y2 = box
        w_px = x2 - x1
        h_px = y2 - y1
        diameter_px = (w_px + h_px) / 2.0
        radius_px = diameter_px / 2.0
        area_px2 = np.pi * (radius_px ** 2)

        diameter_um = diameter_px * PIXEL_SIZE_UM
        radius_um = radius_px * PIXEL_SIZE_UM
        area_um2 = area_px2 * (PIXEL_SIZE_UM ** 2)

        records_all.append({
            "source": "full_dataset_pred",
            "image": img_path.name,
            "class_id": cls_id,
            "confidence": conf,
            "width_px": w_px,
            "height_px": h_px,
            "diameter_px": diameter_px,
            "radius_px": radius_px,
            "area_px2": area_px2,
            "diameter_um": diameter_um,
            "radius_um": radius_um,
            "area_um2": area_um2,
        })

if not records_all:
    print("No detections on full dataset.")
else:
    df_all_pred = pd.DataFrame(records_all)
    print(f"Detections (full dataset): {len(df_all_pred)}")

    # Save combined CSV if you want
    out_csv = "vesicle_sizes_full_dataset.csv"
    df_all_pred.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

    plt.figure(figsize=(6,4))
    plt.hist(df_all_pred["diameter_px"], bins=30)
    plt.xlabel("Vesicle diameter (pixels)")
    plt.ylabel("Count")
    plt.title("Size distribution – YOLO predictions (full dataset)")
    plt.tight_layout()
    plt.show()


# Color Analysis

# Imbedded GUI